# Entregable reproducible: TransMilenio × NUSE

Producto analítico del DataJAM (numeral 3.2). El  código deja evidencia de **limpieza**, **transformación** e **integración** de al menos dos fuentes públicas.

**Problema.** El patrón de uso de la troncal (volumen, noche, finde, picos) ¿se asocia a más demanda de seguridad en el *entorno* de las estaciones?

**Unidad correcta.** El flujo es de la estación. El 123 (NUSE) llega por UPZ. No se finge hurto en el andén.

Para reconstruir el panel completo desde crudo (varios minutos, usa red la primera vez):
`python integracion_panel.py` → `python analisis_upz.py` → `python modelo_predictivo.py`.

## Fuentes públicas (más de dos)

1. **TransMilenio** — validaciones (`troncal_YYYY.csv`) y salidas (`salidas_YYYY.csv`), datos abiertos de demanda troncal.
2. **NUSE línea 123** — llamadas tramitadas C4, Datos Abiertos Bogotá (CSV con `;`).
3. Extra del mismo portal: población SDS por localidad, delito de alto impacto, incidente reportado y CAI (ZIP/CSV; nada de `oaiee.scj.gov.co`).

Dependencias: ver `requirements.txt`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path(".")
print("Carpeta:", BASE.resolve())

Carpeta: C:\Users\camilo.raba\Downloads\DataJAM\DataJAM


## 1. Limpieza

Tres reglas que el merge viejo no cumplía:

- `fillna(0)` **antes** de sumar validaciones + salidas (un NaN no debe borrar el flujo).
- Ventanas horarias **disjuntas**: pico AM 06–08, pico PM 17–18, nocturno 19–05 (la hora 19 no entra en las dos).
- Fuera de análisis: Soacha (sin UPZ Bogotá) y meses con menos de 20 días de dato.

In [2]:
# Catálogo de estaciones y flag Soacha (join espacial ya viene en Dim_estaciones.UPLCODIGO).
est = pd.read_csv(BASE / "Dim_estaciones.csv")
est["upz_id"] = est["UPLCODIGO"].astype(str).str.strip()
est["sin_upz"] = est["upz_id"].isin(["", "nan", "None"]) | est["upz_id"].isna()
soacha = {"leon xiii", "terreros", "la despensa", "san mateo"}
est["nom_norm"] = (
    est["nom_est"].astype(str).str.lower()
    .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
)
est["flag_soacha"] = est["nom_norm"].str.contains("leon xiii|terreros|despensa|san mateo", regex=True) | est["sin_upz"]

print("Estaciones en catálogo:", len(est), "| sin UPZ Bogotá:", int(est["flag_soacha"].sum()))

# Demostración de fillna(0): si una hora tiene validaciones y la otra no, el flujo no se pierde.
demo = pd.DataFrame({"validaciones": [10.0, np.nan], "salidas": [np.nan, 5.0]})
demo["flujo_mal"] = demo["validaciones"] + demo["salidas"]          # NaN contagia
demo["flujo_bien"] = demo["validaciones"].fillna(0) + demo["salidas"].fillna(0)
demo

Estaciones en catálogo: 150 | sin UPZ Bogotá: 4


,validaciones,salidas,flujo_mal,flujo_bien
0,10.0,NaN,NaN,10.0
1,NaN,5.0,NaN,5.0


## 2. Transformación

NUSE trae decenas de `TIPO_DETALLE`. Los agrupamos en familias (hurto / violencia / orden público). El resto queda en `otros`.

In [3]:
FAMILIAS = {
    "hurto": {"HURTO EFECTUADO", "HURTO EN PROCESO"},
    "violencia": {
        "RIÑA", "LESIONES PERSONALES", "DISPAROS", "HERIDO", "VIOLENCIA SEXUAL",
        "MALTRATO", "PORTE DE ARMAS", "RAPTO", "SECUESTRO",
    },
    "orden_publico": {"EMBRIAGUEZ", "HABITANTE DE LA CALLE", "PANDILLAS", "RUIDO", "SOSPECHOSO"},
}

crudos = list(BASE.glob("llamadastramitadas*.csv"))
if not crudos:
    print("Sin CSV crudo de NUSE en esta copia (no cabe en GitHub).")
    print("El panel ya integrado está en salidas_integracion/; sigue a la celda de Integración.")
else:
    nuse_path = crudos[0]
    # Muestra (el archivo completo tiene >1 millón de filas; el pipeline usa todo).
    nuse = pd.read_csv(nuse_path, sep=";", nrows=8000, dtype=str)
    nuse["upz_id"] = nuse["COD_UPZ"].astype(str).str.strip()
    nuse["cant"] = pd.to_numeric(nuse["CANT_INCIDENTES"], errors="coerce").fillna(0)

    def familia(tipo):
        t = str(tipo).strip().upper()
        for nom, s in FAMILIAS.items():
            if t in s:
                return nom
        return "otros"

    nuse["familia"] = nuse["TIPO_DETALLE"].map(familia)
    nuse = nuse.loc[~nuse["upz_id"].isin(["UPZ999", ""])]
    print(nuse["familia"].value_counts().head())
    display(nuse.groupby(["upz_id", "familia"], as_index=False)["cant"].sum().head())

familia
otros            5261
violencia         880
orden_publico     332
hurto             218
Name: count, dtype: int64


,upz_id,familia,cant
0,UPR1,orden_publico,2
1,UPR1,otros,49
2,UPR1,violencia,13
3,UPR3,hurto,4
4,UPR3,orden_publico,13


## 3. Integración

Cruce: estación–mes (TM) ⋈ UPZ (catálogo) ⋈ NUSE agregado a UPZ–mes. Los outcomes llevan prefijo `upz_` o `loc_` para no leerlos como hechos de la estación.

El panel ya integrado está en `salidas_integracion/panel_estacion_mes.csv` (tras `integracion_panel.py`).

In [4]:
panel = pd.read_csv(BASE / "salidas_integracion" / "panel_estacion_mes.csv")
upz = pd.read_csv(BASE / "salidas_analisis" / "panel_upz_mes_agregado.csv")
nombres = pd.read_csv(BASE / "salidas_analisis" / "nombres_upz.csv")
nombres["upz_id"] = nombres["upz_id"].astype(str).str.strip()

# Integración explícita: features TM de estación → una fila por UPZ–mes + nombre de la UPZ.
upz["upz_id"] = upz["upz_id"].astype(str).str.strip()
cruz = upz.merge(nombres, on="upz_id", how="left")

print("Panel estación–mes:", panel.shape, "| estaciones", panel["cod_estacion"].nunique())
print("Panel UPZ–mes:", cruz.shape, "| UPZ", cruz["upz_id"].nunique())
print("Columnas de entorno (no de estación):", [c for c in panel.columns if c.startswith("upz_nuse_")])
cruz.loc[cruz["anio"] == 2023, ["upz_id", "upz_nombre", "localidad", "flujo_total_upz", "upz_nuse_hurto"]].head()

Panel estación–mes: (6018, 76) | estaciones 149
Panel UPZ–mes: (2434, 35) | UPZ 58
Columnas de entorno (no de estación): ['upz_nuse_hurto', 'upz_nuse_violencia', 'upz_nuse_orden_publico', 'upz_nuse_otros', 'upz_nuse_total']


,upz_id,upz_nombre,localidad,flujo_total_upz,upz_nuse_hurto
0,UPZ10,LA URIBE,USAQUÉN,3465755.0,29.0
1,UPZ10,LA URIBE,USAQUÉN,3983681.0,39.0
2,UPZ10,LA URIBE,USAQUÉN,4489185.0,31.0
3,UPZ10,LA URIBE,USAQUÉN,3868303.0,32.0
4,UPZ10,LA URIBE,USAQUÉN,4322832.0,43.0


## Hallazgos (coherentes con el dashboard)

1. Hurto al 123: La Sabana (centro). El % nocturno no ordena las UPZ (ρ ≈ 0,03).
2. Violencia: El Rincón y Bosa Central. Sí se parece al fin de semana (ρ ≈ 0,51).
3. Pronóstico 2026 (bosque vs repetir el mes previo): +12 % MAE en hurto, +23 % en violencia. El 123 rezagado manda; el perfil TM aporta poco.

Visualización: `python -m shiny run dashboard/app.py --port 8000`. Informe compartible: `python dashboard/exportar_informe.py`.

In [5]:
rank = pd.read_csv(BASE / "salidas_analisis" / "ranking_upz.csv")
rank = rank.merge(nombres, on="upz_id", how="left")
print("Top hurto (conteo mensual medio)")
print(rank.nsmallest(5, "rank_hurto_medio")[["upz_nombre", "localidad", "hurto_mensual_medio", "n_estaciones"]].to_string(index=False))

met = pd.read_csv(BASE / "salidas_modelo" / "metricas.csv")
print("\nModelo 2026")
print(met.loc[met["modelo"].isin(["bosque", "persistencia (mes previo)"])][
    ["familia", "modelo", "mae", "skill_vs_persistencia"]
].to_string(index=False))

Top hurto (conteo mensual medio)
   upz_nombre      localidad  hurto_mensual_medio  n_estaciones
    LA SABANA   LOS MÁRTIRES           314.333333           8.0
    EL RINCON           SUBA           289.976190           2.0
   LAS FERIAS       ENGATIVÁ           258.714286           3.0
LOS ALCAZARES BARRIOS UNIDOS           241.619048           3.0
   CHICO LAGO      CHAPINERO           240.380952           6.5

Modelo 2026
  familia                    modelo       mae  skill_vs_persistencia
    hurto persistencia (mes previo) 19.023121               0.000000
    hurto                    bosque 16.712088               0.121486
violencia persistencia (mes previo) 66.381503               0.000000
violencia                    bosque 51.363875               0.226232
